In [44]:
import math
import random
import json

In [45]:
def extended_gcd(a, b):
    sa = 1 if a >= 0 else -1
    sb = 1 if b >= 0 else -1
    a, b = abs(a), abs(b)
    
    if a == 0:
        return b, 0, sb
    
    g, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    
    return g, x * sa, y * sb

def mod_inverse(a, m):
    a %= m 
    g, x, _ = extended_gcd(a, m)
    
    return x % m if g==1 else None

def pow_mod(base, exp, mod):
    result = 1
    base %= mod
    while exp > 0:
        if exp % 2 == 1:
            result = (result * base) % mod
        exp //= 2
        base = (base * base) % mod
    return result

In [46]:
def check_math_menu():
    print("\n=== ПРОВЕРКА МАТЕМАТИКИ ===")
    choice = input("1. Алгоритм Евклида  2. Возведение в степень  0. Назад\nВыбор: ").strip()
    if choice == '1':
        try:
            a = int(input("Введите число a: "))
            b = int(input("Введите число b: "))

            g, x, y = extended_gcd(a, b)

            print(f"Вход: a={a}, b={b}")
            print(f"НОД({a}, {b}) = {g}")
            print(f"Коэффициенты Безу: x={x}, y={y}")
            print(f"Проверка тождества Безу: {a}*({x}) + {b}*({y}) = {a * x + b * y}")

        except ValueError:
            print("Ошибка ввода: необходимо ввести целые числа.")

    elif choice == '2':
        try:
            base = int(input("Введите основание (base): "))
            exp = int(input("Введите показатель степени (exp): "))
            mod = int(input("Введите модуль (mod): "))

            if mod <= 0:
                print("Ошибка: модуль должен быть положительным числом.")
                return

            result = pow_mod(base, exp, mod)

            print(f"{base}^{exp} mod {mod} = {result}")

        except ValueError:
            print("Ошибка ввода: необходимо ввести целые числа.")

    elif choice == '0':
        print("Возврат в главное меню.")
        return False

    else:
        print("Неверный выбор. Попробуйте снова.")

    return True

In [47]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True

def generate_random_e(phi_n):
    while True:
        e = random.randrange(3, phi_n, 2)
        if math.gcd(e, phi_n) == 1:
            return e

def generate_rsa_keypair(p, q, e):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError("p и q должны быть простыми")
    
    n = p * q
    phi_n = (p - 1) * (q - 1)
    
    if math.gcd(e, phi_n) != 1:
        raise ValueError(f"e={e} не взаимно просто с φ(n)")
        
    d = mod_inverse(e, phi_n)
    if d is None:
        raise ValueError("Не удалось найти обратный элемент d")
        
    return {
        'n': n, 
        'e': e, 
        'd': d
    }


In [48]:
def generate_keys_menu():
    print("\n=== ГЕНЕРАЦИЯ КЛЮЧЕЙ RSA ===")
    
    p_input = input("Введите простое число p [по умолч.: 227]: ").strip()
    p = int(p_input) if p_input else 227
    
    q_input = input("Введите простое число q [по умолч.: 373]: ").strip()
    q = int(q_input) if q_input else 373
    
    keys = []
    try:
        for i in range(1, 4):
            print(f"\n--- Генерация пары ключей #{i} ---")
            
            e_str = input(f"Введите открытую экспоненту e для пары #{i} [Enter = случайно]: ").strip()
            
            if not e_str:
                phi_n_temp = (p - 1) * (q - 1)
                e = generate_random_e(phi_n_temp)
                print(f"Случайно сгенерировано e = {e}")
            else:
                e = int(e_str)
            
            kp = generate_rsa_keypair(p, q, e)
            keys.append(kp)
            
            print(f"Пара #{i}: e={kp['e']}, d={kp['d']}, n={kp['n']}")
            
        filename = "rsa_keys.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump({
                'params': {'p': p, 'q': q},
                'keys': keys
            }, f, indent=2, ensure_ascii=False)
                
        print(f"\nКлючи сохранены в '{filename}'")
        
    except ValueError as ve:
        print(f"Ошибка генерации: {ve}")
    except Exception as ex:
        print(f"Непредвиденная ошибка: {ex}")

In [49]:
ALPHABET_RU = "АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"
CHAR_TO_CODE = {char: str(10 + i) for i, char in enumerate(ALPHABET_RU)}
CHAR_TO_CODE[' '] = '99'

def prepare_text(text):
    """Подготовка открытого текста: верхний регистр, только русские буквы и пробелы"""
    cleaned = ""
    for char in text.upper():
        if char in ALPHABET_RU or char == ' ':
            cleaned += char
    return cleaned

def text_to_numeric_string(text):
    """Преобразование текста в строку цифровых кодов"""
    numeric_parts = []
    for char in text:
        code = CHAR_TO_CODE.get(char)
        if code is not None:
            numeric_parts.append(code)
    return "".join(numeric_parts)

In [50]:
def split_into_blocks(numeric_str, n):
    """Разбивает числовую строку на блоки, каждый из которых меньше n.

    Блок не может начинаться с нуля: иначе при обратном склеивании
    (int -> str) ведущий ноль потеряется и текст не восстановится.
    """
    if n <= 1:
        raise ValueError("Модуль n должен быть больше 1")

    blocks = []
    max_len = len(str(n))
    i = 0
    while i < len(numeric_str):
        block_len = min(max_len, len(numeric_str) - i)
        while block_len > 1:
            candidate = numeric_str[i:i + block_len]
            next_idx = i + block_len
            if int(candidate) < n and (next_idx >= len(numeric_str) or numeric_str[next_idx] != '0'):
                break
            block_len -= 1
        blocks.append(int(numeric_str[i:i + block_len]))
        i += block_len

    if "".join(str(b) for b in blocks) != numeric_str:
        raise ValueError(
            "Модуль n слишком мал для двузначных кодов: "
            "разбиение теряет ведущие нули"
        )

    return blocks

In [51]:
def encrypt_menu():
    print("\n=== ШИФРОВАНИЕ RSA ===")
    try:
        with open("rsa_keys.json", encoding="utf-8") as f:
            keys = json.load(f)["keys"]
    except (FileNotFoundError, KeyError, ValueError):
        keys = []

    e = n = None
    if keys:
        key_list = "\n".join(
            f"{i}. e={k['e']}, n={k['n']}"
            for i, k in enumerate(keys, 1)
        )
        choice = input(f"Доступные ключи:\n{key_list}\nНомер ключа [Enter = ввести свой]: ").strip()
        if choice:
            try:
                idx = int(choice) - 1
                if 0 <= idx < len(keys):
                    e, n = keys[idx]['e'], keys[idx]['n']
                else:
                    print("Неверный номер ключа.")
                    return
            except ValueError:
                print("Ошибка ввода номера.")
                return
    else:
        print("Файл ключей не найден — введите ключ вручную.")

    if e is None:
        try:
            e = int(input("Введите e: "))
            n = int(input("Введите n: "))
        except ValueError:
            print("Ошибка ввода: нужны целые числа.")
            return

    print(f"Ключ: e={e}, n={n}")
    text = prepare_text(input("Открытый текст: "))
    if not text:
        print("Текст пуст.")
        return

    try:
        blocks = split_into_blocks(text_to_numeric_string(text), n)
    except ValueError as ve:
        print(f"Ошибка: {ve}")
        return

    print(f"Блоки открытого текста: {blocks}")

    block_len = len(str(n))
    cipher = [pow_mod(m, e, n) for m in blocks]
    print(f"ШИФР-ТЕКСТ: {''.join(str(c).zfill(block_len) for c in cipher)}")

In [ ]:
CODE_TO_CHAR = {v: k for k, v in CHAR_TO_CODE.items()}
VALID_CODES = set(CHAR_TO_CODE.values())
VALID_FIRST_DIGITS = {code[0] for code in VALID_CODES}


def prefix_ok(digits):
    """Проверяет, что строка цифр может быть началом последовательности кодов."""
    for i in range(0, len(digits) - 1, 2):
        if digits[i:i + 2] not in VALID_CODES:
            return False
    if len(digits) % 2 == 1 and digits[-1] not in VALID_FIRST_DIGITS:
        return False
    return True


def find_cipher_blocks(cipher_str, d, n):
    """Подбирает границы блоков шифр-текста перебором с возвратом.

    Возвращает список расшифрованных блоков или None, если разбиение не найдено.
    """
    max_len = len(str(n))
    decrypted = []
    dead_ends = set()

    def search(pos, digits):
        if pos == len(cipher_str):
            return len(digits) % 2 == 0
        state = (pos, digits[-1] if len(digits) % 2 else '')
        if state in dead_ends:
            return False
        limit = min(max_len, len(cipher_str) - pos)
        for length in range(limit, 0, -1):
            chunk = cipher_str[pos:pos + length]
            value = int(chunk)
            if value >= n:
                continue
            block = pow_mod(value, d, n)
            candidate = digits + str(block)
            if not prefix_ok(candidate):
                continue
            decrypted.append(block)
            if search(pos + length, candidate):
                return True
            decrypted.pop()
        dead_ends.add(state)
        return False

    return decrypted if search(0, "") else None


def decrypt_blocks_to_text(decrypted_blocks):
    """Склеивает блоки и разбирает цифровую строку по парам кодов."""
    numeric_str = "".join(str(b) for b in decrypted_blocks)
    text = ""
    for i in range(0, len(numeric_str) - 1, 2):
        text += CODE_TO_CHAR.get(numeric_str[i:i + 2], '?')
    return text


In [ ]:
def decrypt_menu():
    print("\n=== РАСШИФРОВАНИЕ RSA ===")
    try:
        with open("rsa_keys.json", encoding="utf-8") as f:
            keys = json.load(f)["keys"]
    except (FileNotFoundError, KeyError, ValueError):
        keys = []

    d = n = None
    if keys:
        key_list = "\n".join(
            f"{i}. d={k['d']}, n={k['n']}"
            for i, k in enumerate(keys, 1)
        )
        choice = input(f"Доступные ключи:\n{key_list}\nНомер ключа [Enter = ввести свой]: ").strip()
        if choice:
            try:
                idx = int(choice) - 1
                if 0 <= idx < len(keys):
                    d, n = keys[idx]['d'], keys[idx]['n']
                else:
                    print("Неверный номер ключа.")
                    return
            except ValueError:
                print("Ошибка ввода номера.")
                return
    else:
        print("Файл ключей не найден — введите ключ вручную.")

    if d is None:
        try:
            d = int(input("Введите d: "))
            n = int(input("Введите n: "))
        except ValueError:
            print("Ошибка ввода: нужны целые числа.")
            return

    print(f"Ключ: d={d}, n={n}")
    s = input("Шифр-текст (одной строкой цифр): ").strip()
    if not s:
        print("Шифр-текст пуст.")
        return
    if not s.isdigit():
        print("Ошибка: шифр-текст должен состоять только из цифр.")
        return

    blocks = find_cipher_blocks(s, d, n)
    if blocks is None:
        print("Не удалось однозначно разбить шифр-текст на блоки.")
        return

    print(f"Расшифрованные блоки: {blocks}")
    text = decrypt_blocks_to_text(blocks)
    print(f"ОТКРЫТЫЙ ТЕКСТ: {text}")


In [ ]:
def crypto_menu():
    while True:
        print("\n=== ШИФРОВАНИЕ / РАСШИФРОВАНИЕ ===")
        choice = input("1. Зашифровать  2. Расшифровать  0. Назад\nВыбор: ").strip()

        if choice == '1':
            encrypt_menu()
        elif choice == '2':
            decrypt_menu()
        elif choice == '0':
            break
        else:
            print("Неверный выбор, попробуйте снова.")


def main_menu():
    while True:
        print("\n=== ГЛАВНОЕ МЕНЮ (RSA) ===")
        choice = input("1. Проверка математики  2. Генерация ключей  "
                       "3. Шифрование/расшифрование  0. Выход\nВыбор: ").strip()

        if choice == '1':
            while check_math_menu():
                pass
        elif choice == '2':
            generate_keys_menu()
        elif choice == '3':
            crypto_menu()
        elif choice == '0':
            print("Выход из программы.")
            break
        else:
            print("Неверный выбор, попробуйте снова.")


main_menu()